# А/В-тест для «Иголочка и ниточка»

В ноутбуке собраны ответы и расчеты по заданиям:
1. Выбор гипотезы и ключевой метрики.
2. Расчет ожидаемого эффекта (MDE) на исторических данных.
3. Расчет размера выборки и длительности эксперимента.
4. Проверка статистической значимости на итоговых данных.

## Задание 1. Анализ гипотез и выбор метрик

**Выбранная гипотеза (в первую очередь):**
Наличие видеообзора товара на странице может помочь покупателю лучше понять характеристики и особенности продукта.

Обоснование: у этой гипотезы лучший баланс по ICE-подобной логике (высокий потенциальный эффект = 9, достаточная простота = 7, невысокая стоимость = 4).

**Ключевая метрика эксперимента:**
Конверсия — процент пользователей, выполнивших целевое действие.

In [1]:
import math
from pathlib import Path

import pandas as pd
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest

# Данные лежат в корне проекта в папке misc
base = Path("../../misc")
opened_normal = pd.read_csv(base / "ab_mde_opened_normal.csv")
clicked_normal = pd.read_csv(base / "ab_mde_clicked_normal.csv")
opened_sale = pd.read_csv(base / "ab_mde_opened_sale.csv")
clicked_sale = pd.read_csv(base / "ab_mde_clicked_sale.csv")

cr_normal = len(clicked_normal) / len(opened_normal)
cr_sale = len(clicked_sale) / len(opened_sale)

# Абсолютный ожидаемый эффект в процентных пунктах
mde_pp = (cr_sale - cr_normal) * 100
mde_abs_int = round(mde_pp)

print(f"CR normal: {cr_normal:.4%}")
print(f"CR sale:   {cr_sale:.4%}")
print(f"MDE (pp):  {mde_pp:.3f}")
print(f"MDE (целое абсолютное): {mde_abs_int}")

CR normal: 9.4883%
CR sale:   14.4804%
MDE (pp):  4.992
MDE (целое абсолютное): 5


## Задание 2. Расчет ожидаемого эффекта (MDE)

По историческим данным:
- CR в обычный период: ~9.49%
- CR в период акции: ~14.48%
- Абсолютный эффект: ~4.99 п.п.

**Ответ (MDE, целое абсолютное значение): 5**

In [2]:
# Параметры эксперимента для расчета размера выборки
alpha = 0.05
power = 0.80
traffic_per_day = 240  # суммарный трафик в день на обе группы

effect_size = abs(proportion_effectsize(cr_normal, cr_sale))
n_per_group = NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative="two-sided",
)

n_per_group_ceil = math.ceil(n_per_group)
n_total = n_per_group_ceil * 2
days = math.ceil(n_total / traffic_per_day)

print(f"Размер выборки одной группы: {n_per_group_ceil}")
print(f"Общий размер выборки:         {n_total}")
print(f"Длительность эксперимента:    {days} дней")

Размер выборки одной группы: 658
Общий размер выборки:         1316
Длительность эксперимента:    6 дней


## Задание 3. Расчет параметров эксперимента

При $\alpha=0.05$, мощности 80% и ожидаемом эффекте из задания 2:

- **Объем выборки одной группы:** 658 чел.
- **Общий объем выборки:** 1316 чел.
- **Количество дней эксперимента:** 6

In [3]:
results = pd.read_csv(base / "ab_results_1.csv")
agg = results.groupby("group")["converted"].agg(["sum", "count", "mean"])
display(agg)

control_success = int(agg.loc["control", "sum"])
control_n = int(agg.loc["control", "count"])
treat_success = int(agg.loc["treatment", "sum"])
treat_n = int(agg.loc["treatment", "count"])

z_stat, p_value = proportions_ztest(
    [control_success, treat_success],
    [control_n, treat_n],
    alternative="two-sided",
)

print(f"z-stat: {z_stat:.3f}")
print(f"p-value: {p_value:.6f}")
print(f"p-value (до 3 знаков): {p_value:.3f}")
print(f"CR control:   {control_success/control_n:.3%}")
print(f"CR treatment: {treat_success/treat_n:.3%}")

,sum,count,mean
group,,,
control,61,613,0.099511
treatment,81,592,0.136824


z-stat: -2.008
p-value: 0.044605
p-value (до 3 знаков): 0.045
CR control:   9.951%
CR treatment: 13.682%


## Задание 4. Подведение итогов A/B-теста

- p-value по результатам эксперимента: **0.045**
- При уровне значимости 0.05 различия статистически значимы.
- Конверсия в группе B (`treatment`) выше, чем в группе A (`control`).

**Ответ на вопрос 6:** Вариант В является более предпочтительным.